# Random Forest Baseline for PHA Classification


In [1]:
# Imports and load the exact project dataset from GitHub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)

DATA_URL = (
    "https://raw.githubusercontent.com/meherhasija/"
    "neo-missingness-lab/refs/heads/main/"
    "data/neo_asteroid_dataset.csv"
)

df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
print("Dataset loaded directly from GitHub.")
display(df.head())


Dataset shape: (42815, 21)
Dataset loaded directly from GitHub.


,spkid,pdes,full_name,kind,neo,pha,e,a,q,i,...,w,moid,H,diameter,albedo,first_obs,last_obs,data_arc,n_obs_used,condition_code
0,20000433,433,433 Eros (A898 PA),an,Y,N,0.2229,1.458,1.133,10.83,...,178.92,0.1490,10.40,16.840,0.250,1893-10-29,2021-05-13,46582.0,9130,0.0
1,20000719,719,719 Albert (A911 TB),an,Y,N,0.5466,2.637,1.196,11.57,...,156.18,0.2020,15.59,NaN,NaN,1911-10-04,2026-05-22,41869.0,2147,0.0
2,20000887,887,887 Alinda (A918 AA),an,Y,N,0.5711,2.474,1.061,9.40,...,350.55,0.0801,13.82,4.200,0.310,1918-02-09,2025-08-27,39281.0,5843,0.0
3,20001036,1036,1036 Ganymed (A924 UB),an,Y,N,0.5335,2.664,1.243,26.69,...,132.51,0.3420,9.17,37.675,0.238,1924-10-23,2026-06-30,37140.0,13637,0.0
4,20001221,1221,1221 Amor (1932 EA1),an,Y,N,0.4347,1.920,1.085,11.87,...,26.75,0.1090,17.37,1.000,NaN,1932-03-12,2026-10-20,34555.0,737,0.0


In [2]:
# Dataset audit
print("Original rows:", len(df))
print("\\nPHA values including missing:")
print(df["pha"].value_counts(dropna=False))

df["pha"] = df["pha"].astype(str).str.strip().str.upper()
df = df[df["pha"].isin(["Y", "N"])].copy()

print("\\nLabelled rows:", len(df))
print("\\nPHA class counts:")
print(df["pha"].value_counts())
print("\\nPHA class percentages:")
print((df["pha"].value_counts(normalize=True) * 100).round(2))
print("\\nDiameter missing:", df["diameter"].isna().sum())
print("Diameter missing percentage:", round(df["diameter"].isna().mean() * 100, 2), "%")

Original rows: 42815
\nPHA values including missing:
pha
N      40130
Y       2550
NaN      135
Name: count, dtype: int64
\nLabelled rows: 42680
\nPHA class counts:
pha
N    40130
Y     2550
Name: count, dtype: int64
\nPHA class percentages:
pha
N    94.03
Y     5.97
Name: proportion, dtype: float64
\nDiameter missing: 41434
Diameter missing percentage: 97.08 %


In [3]:
# Feature selection and leakage precautions
# Selected features:
# - a: semi-major axis
# - e: eccentricity
# - i: inclination
# - om: longitude of ascending node
# - w: argument of perihelion
# - diameter: effective diameter
FEATURES = ["a", "e", "i", "om", "w", "diameter"]
TARGET = "pha"

X = df[FEATURES].copy()
y = df[TARGET].map({"N": 0, "Y": 1}).astype(int)

print("Features:", FEATURES)
print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Features: ['a', 'e', 'i', 'om', 'w', 'diameter']
X shape: (42680, 6)
Target distribution:
pha
0    40130
1     2550
Name: count, dtype: int64


In [4]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\\nPHA rate in training set:", round(y_train.mean() * 100, 2), "%")
print("PHA rate in test set:", round(y_test.mean() * 100, 2), "%")

Training rows: 34144
Testing rows: 8536
\nPHA rate in training set: 5.97 %
PHA rate in test set: 5.97 %


In [5]:
# ============================================================
# RANDOM FOREST: FAIR THRESHOLD TUNING USING 5-FOLD OOF
# ============================================================
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve
)

MIN_PRECISION = 0.20
N_FOLDS = 5
RANDOM_STATE = 42

# The common split is exactly the same as the main notebook.
def create_rf_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("random_forest", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

print("Generating out-of-fold probabilities from training data...")
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_probabilities = cross_val_predict(
    create_rf_pipeline(), X_train, y_train,
    cv=cv, method="predict_proba", n_jobs=1
)[:, 1]

precision_values, recall_values, thresholds = precision_recall_curve(
    y_train, oof_probabilities
)
valid_indices = np.where(precision_values[:-1] >= MIN_PRECISION)[0]
if len(valid_indices) == 0:
    raise ValueError("No threshold meets the minimum precision requirement.")

# Maximise recall among thresholds meeting the training-only precision constraint.
best_index = max(valid_indices, key=lambda i: (recall_values[i], precision_values[i]))
best_threshold = float(thresholds[best_index])
print("Selected threshold:", round(best_threshold, 4))
print("OOF precision:", round(float(precision_values[best_index]), 4))
print("OOF recall:", round(float(recall_values[best_index]), 4))

# Refit once on all training records, then evaluate on untouched test records.
rf_final_model = create_rf_pipeline()
rf_final_model.fit(X_train, y_train)
test_probabilities = rf_final_model.predict_proba(X_test)[:, 1]
default_predictions = (test_probabilities >= 0.50).astype(int)
tuned_predictions = (test_probabilities >= best_threshold).astype(int)

def rf_metrics(actual, predicted, probabilities, method_name):
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {
        "Method": method_name,
        "Records": len(actual),
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1": f1_score(actual, predicted, zero_division=0),
        "ROC_AUC": roc_auc_score(actual, probabilities),
        "Average_Precision": average_precision_score(actual, probabilities),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

rf_results = pd.DataFrame([
    rf_metrics(y_test, default_predictions, test_probabilities, "RF_Default_Threshold_0.50"),
    rf_metrics(y_test, tuned_predictions, test_probabilities, "RF_OOF_Tuned_Threshold")
])
print("\nRANDOM FOREST TEST RESULTS")
display(rf_results.round(4))
print("\nDefault confusion matrix:")
print(confusion_matrix(y_test, default_predictions, labels=[0, 1]))
print("\nTuned confusion matrix:")
print(confusion_matrix(y_test, tuned_predictions, labels=[0, 1]))

rf_predictions = pd.DataFrame({
    "original_row_index": X_test.index,
    "actual_pha": y_test.values,
    "predicted_probability": test_probabilities,
    "default_prediction": default_predictions,
    "tuned_prediction": tuned_predictions
})
threshold_info = pd.DataFrame([{
    "threshold_selection": "5-fold stratified out-of-fold predictions on training data",
    "minimum_precision_constraint": MIN_PRECISION,
    "selected_threshold": best_threshold,
    "OOF_precision_at_selected_threshold": precision_values[best_index],
    "OOF_recall_at_selected_threshold": recall_values[best_index],
    "training_records": len(X_train),
    "test_records": len(X_test),
    "random_state": RANDOM_STATE
}])

OUTPUT_DIR = "/content/neo_evaluation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
rf_results.to_csv(os.path.join(OUTPUT_DIR, "random_forest_fair_comparison_metrics.csv"), index=False)
rf_predictions.to_csv(os.path.join(OUTPUT_DIR, "random_forest_fair_comparison_predictions.csv"), index=False)
threshold_info.to_csv(os.path.join(OUTPUT_DIR, "random_forest_fair_selected_threshold.csv"), index=False)

# Package only the three Random Forest outputs for easy GitHub upload.
import zipfile
zip_path = "/content/Random_Forest_Results.zip"
rf_files = [
    "random_forest_fair_comparison_metrics.csv",
    "random_forest_fair_comparison_predictions.csv",
    "random_forest_fair_selected_threshold.csv"
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in rf_files:
        zf.write(os.path.join(OUTPUT_DIR, filename), arcname=filename)

print("\nSaved Random Forest CSV files in:", OUTPUT_DIR)
print("Downloading:", zip_path)
from google.colab import files
files.download(zip_path)


Generating out-of-fold probabilities from training data...
Selected threshold: 0.1233
OOF precision: 0.2026
OOF recall: 0.4838

RANDOM FOREST TEST RESULTS


,Method,Records,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision,TN,FP,FN,TP
0,RF_Default_Threshold_0.50,8536,0.9420,0.8571,0.0353,0.0678,0.7983,0.2658,8023,3,492,18
1,RF_OOF_Tuned_Threshold,8536,0.8653,0.2178,0.4843,0.3005,0.7983,0.2658,7139,887,263,247



Default confusion matrix:
[[8023    3]
 [ 492   18]]

Tuned confusion matrix:
[[7139  887]
 [ 263  247]]

Saved Random Forest CSV files in: /content/neo_evaluation_results
Downloading: /content/Random_Forest_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>